In [15]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.append("/sujin/PycharmProjects/rllm")

import json

from pathlib import Path
from rllm.data import DatasetRegistry, Dataset
from rllm.eval.agent_loader import load_agent, register_agent, list_agents
from rllm.runner import Runner
from rllm.types import AgentConfig, Task
from rllm.eval.evaluator_loader import load_evaluator, list_evaluators, register_evaluator
from tqdm import tqdm
from utils.mpr import MultipleProcessRunnerSimplifier

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Tokenizer check

In [84]:
from transformers import AutoTokenizer

_CALCULATOR_DESCRIPTION = (
    "Evaluate a mathematical expression. Supports operators +, -, *, /, //, ** (power), "
    "% (modulo), and parentheses. Functions: abs, round, min, max, sum, pow, sqrt, exp, "
    "log, log2, log10, sin, cos, tan, asin, acos, atan, atan2, sinh, cosh, tanh, floor, "
    "ceil, trunc, factorial, gcd, lcm, comb (aka binom), perm, degrees, radians. "
    "Constants: pi, e, tau."
)

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": _CALCULATOR_DESCRIPTION,
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "The expression to evaluate, e.g. 'sqrt(64 + 225)' or 'binom(10, 3) * pi'",
                    }
                },
                "required": ["expression"],
            },
        },
    }
]

model_name = "/sujin/Models/Qwen/Qwen3-4B"

SYSTEM_PROMPT = """\
You are a math assistant that solves competition math problems step by step.
You have access to a calculator tool and you MUST use it.

IMPORTANT rules you must follow:
1. You MUST call the calculator tool at least once before giving your final answer. \
Answers given without any prior tool call will be marked wrong.
2. Do NOT perform arithmetic in your head — every computation must go through the calculator.
3. Break the problem into small steps. Make one tool call per step, then reason about the result.
4. When you have the final answer, put it in \\boxed{ANSWER} in your response.
"""

# load the tokenizer and the model
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=False)
# prepare the model input
prompt = "Give me a short introduction to large language model."
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": prompt},
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    enable_thinking=True, # Switches between thinking and non-thinking modes. Default is True.
    tools=TOOLS
)

print(text)
print(len(text))
print(tokenizer.is_fast)

[151644, 8948, 198, 2610, 525, 264, 6888, 17847, 429, 67477, 10707, 6888, 5322, 3019, 553, 3019, 624, 2610, 614, 2615, 311, 264, 29952, 5392, 323, 498, 27732, 990, 432, 382, 98743, 5601, 498, 1969, 1795, 510, 16, 13, 1446, 27732, 1618, 279, 29952, 5392, 518, 3245, 3055, 1573, 7086, 697, 1590, 4226, 13, 37243, 2661, 2041, 894, 4867, 5392, 1618, 686, 387, 12864, 4969, 624, 17, 13, 3155, 4183, 2736, 34784, 304, 697, 1968, 1959, 1449, 34447, 1969, 728, 1526, 279, 29952, 624, 18, 13, 15623, 279, 3491, 1119, 2613, 7354, 13, 7405, 825, 5392, 1618, 817, 3019, 11, 1221, 2874, 911, 279, 1102, 624, 19, 13, 3197, 498, 614, 279, 1590, 4226, 11, 2182, 432, 304, 1124, 79075, 90, 11692, 39351, 92, 304, 697, 2033, 4192, 2, 13852, 271, 2610, 1231, 1618, 825, 476, 803, 5746, 311, 7789, 448, 279, 1196, 3239, 382, 2610, 525, 3897, 448, 729, 32628, 2878, 366, 15918, 1472, 15918, 29, 11874, 9492, 510, 27, 15918, 397, 4913, 1313, 788, 330, 1688, 497, 330, 1688, 788, 5212, 606, 788, 330, 35597, 497, 330, 4684,

In [13]:
def do(process_id, idx, item, writer):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt},
    ]
    text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=True, # Switches between thinking and non-thinking modes. Default is True.
            tools=TOOLS
            )

items = [i for i in range(1000)]
mprs = MultipleProcessRunnerSimplifier(items, do, n_process=256)
mprs.run()

[Errno 25] Inappropriate ioctl for device
Can't get terminal size, set terminal_y = None
7Total: [##########################################] 100% 1000/1000 [00:00:04 < 00:00:00, 222.08it/s]8

Aggregating results...: 256it [00:00, 411080.33it/s]


[]

# Single test

In [103]:
# dataset = DatasetRegistry.load_dataset("deepscaler_math", "train")
# print(len(dataset))
dataset = DatasetRegistry.load_dataset("math500", "test")

item = dataset[123]
print(item.keys())   # → dict_keys(['question', 'answer', ...)
print(item["question"])

# len_prompt = [len(item["question"]) for item in dataset]
# len_prompt = sorted(len_prompt, reverse=True)
# print(len_prompt[:10])
#
# prompt_token_len = []
#
# # Randomly shuffle the dataset to get a more representative sample of prompt token lengths
# from random import shuffle
# dataset = list(dataset)
# shuffle(dataset)
#
# for item in tqdm(dataset):
#     messages = [
#         {"role": "system", "content": SYSTEM_PROMPT},
#         {"role": "user", "content": item["question"]},
#     ]
#     text = tokenizer.apply_chat_template(
#         messages,
#         tokenize=True,
#         add_generation_prompt=True,
#         enable_thinking=True, # Switches between thinking and non-thinking modes. Default is True.
#         tools=TOOLS
#     )
#
#     prompt_token_len.append(len(text))
#     if len(prompt_token_len) == 1000:
#         break
#
# prompt_token_len = sorted(prompt_token_len, reverse=True)
# print(prompt_token_len)

dict_keys(['question', 'ground_truth', 'data_source'])
In the diagram, $D$ and $E$ are the midpoints of $\overline{AB}$ and $\overline{BC}$ respectively.  Determine the area of quadrilateral $DBEF$. [asy]
size(180); defaultpen(linewidth(.7pt)+fontsize(10pt));
pair A, B, C, D, E, F;
A=(0,6);
B=(0,0);
C=(8,0);
D=(0,3);
E=(4,0);
F=(8/3,2);
draw(E--A--C--D);
draw((-1,0)--(10,0), EndArrow);
draw((0,-1)--(0,8), EndArrow);
label("$A(0,6)$", A, W);
label("$B(0,0)$", B, SW);
label("$C(8,0)$", C, S);
label("$D$", D, W);
label("$E$", E, S);
label("$F$", F, SW);
label("$x$", (10,0), dir(0));
label("$y$", (0,8), dir(90));
[/asy]


In [89]:
prompt_token_len = sorted(prompt_token_len, reverse=True)
print(prompt_token_len[:100])

[3707, 1712, 1483, 1380, 1376, 1370, 1327, 1258, 1175, 1129, 1082, 1079, 1076, 1047, 1041, 1015, 1001, 996, 978, 974, 972, 960, 959, 958, 943, 926, 922, 914, 914, 913, 906, 904, 901, 900, 893, 893, 890, 886, 878, 877, 871, 866, 865, 860, 855, 854, 850, 850, 848, 845, 844, 843, 843, 839, 833, 830, 826, 821, 819, 812, 809, 807, 805, 804, 803, 801, 798, 792, 789, 789, 785, 782, 782, 767, 765, 764, 764, 760, 756, 754, 751, 747, 747, 746, 744, 744, 743, 742, 739, 737, 735, 734, 730, 724, 720, 718, 718, 716, 715, 713]


In [68]:
register_agent("math_tool_agent", "cookbooks.math_tool_agent.sync_math_tool_agent:math_tool_agent")
list_agents()

register_evaluator("math_tool_evaluator", "cookbooks.math_tool_agent.evaluator:math_tool_evaluator")
list_evaluators()

[{'name': 'math_tool_evaluator',
  'source': 'registered',
  'type': 'cookbooks.math_tool_agent.evaluator:math_tool_evaluator'},
 {'name': 'bfcl_reward_fn',
  'source': 'built-in',
  'type': 'rllm.eval.reward_fns.bfcl:evaluate'},
 {'name': 'code_reward_fn',
  'source': 'built-in',
  'type': 'rllm.eval.reward_fns.code:evaluate'},
 {'name': 'countdown_reward_fn',
  'source': 'built-in',
  'type': 'rllm.eval.reward_fns.countdown:evaluate'},
 {'name': 'depth_reward_fn',
  'source': 'built-in',
  'type': 'rllm.eval.reward_fns.depth:evaluate'},
 {'name': 'f1_reward_fn',
  'source': 'built-in',
  'type': 'rllm.eval.reward_fns.f1:evaluate'},
 {'name': 'ifeval_reward_fn',
  'source': 'built-in',
  'type': 'rllm.eval.reward_fns.ifeval:evaluate'},
 {'name': 'iou_reward_fn',
  'source': 'built-in',
  'type': 'rllm.eval.reward_fns.iou:evaluate'},
 {'name': 'llm_equality_reward_fn',
  'source': 'built-in',
  'type': 'rllm.eval.reward_fns.llm_equality:evaluate'},
 {'name': 'llm_judge_reward_fn',
  's

In [104]:
agent = load_agent("math_tool_agent")

task = Task(
    id="math500",
    instruction=item["question"] + "\n用中文来回答",
    # instruction="列出你所有可用的工具以及他们的描述给我",
    metadata=item,
    dataset_dir=Path("/root/.rllm/datasets/math500"),  # 相对路径
)

config = AgentConfig(
        base_url="http://localhost:30000/v1",  # LiteLLM proxy 地址
        # model="/sujin/PycharmProjects/rllm/cookbooks/math_tool_agent/checkpoints/math_tool_agent/qwen3-4b-math_tool-verl/global_step_500/actor/huggingface",
        model="/sujin/Models/Qwen/Qwen3-4B",
        session_uid="test_session_uid"
    )

episode = agent.run(task, config)

In [107]:
print("number of trajectories:", len(episode.trajectories))
print("number of steps in the first trajectory:", len(episode.trajectories[0].steps))
print(json.dumps(episode.trajectories[0].steps[-1].chat_completions, indent=4, ensure_ascii=False))

# text = tokenizer.apply_chat_template(
#     episode.trajectories[0].steps[-1].chat_completions,
#     tokenize=False,
#     add_generation_prompt=True,
#     enable_thinking=True, # Switches between thinking and non-thinking modes. Default is True.
#     tools=TOOLS
# )
#
# print(text)

evaluator = load_evaluator("math_tool_evaluator")  # 或其他 registered evaluator
result = evaluator.evaluate(task.metadata, episode)
print(result)

for trajectory in episode.trajectories:
    for step in trajectory.steps:
        print(step)

number of trajectories: 1
number of steps in the first trajectory: 2
[
    {
        "role": "system",
        "content": "You are a math assistant that solves competition math problems step by step.\nYou have access to a calculator tool and you MUST use it.\n\nIMPORTANT rules you must follow:\n1. You MUST call the calculator tool at least once before giving your final answer. Answers given without any prior tool call will be marked wrong.\n2. Do NOT perform arithmetic in your head — every computation must go through the calculator.\n3. Break the problem into small steps. Make one tool call per step, then reason about the result.\n4. When you have the final answer, put it in \\boxed{ANSWER} in your response.\n"
    },
    {
        "role": "user",
        "content": "In the diagram, $D$ and $E$ are the midpoints of $\\overline{AB}$ and $\\overline{BC}$ respectively.  Determine the area of quadrilateral $DBEF$. [asy]\nsize(180); defaultpen(linewidth(.7pt)+fontsize(10pt));\npair A, B, C,

In [63]:
from openai import OpenAI
client = OpenAI(base_url="http://localhost:30000/v1", api_key="EMPTY")
# client = OpenAI(base_url="http://172.98.0.90:37863/v1", api_key="EMPTY")

messages: list[dict] = [
    {"role": "system", "content": ""},
    {"role": "user", "content": "你好"},
]

resp = client.chat.completions.create(
    # model="/sujin/PycharmProjects/rllm/cookbooks/math_tool_agent/checkpoints/math_tool_agent/qwen3-4b-math_tool-verl/global_step_500/actor/huggingface",
    model="/sujin/Models/Qwen/Qwen3-4B",
    messages=messages,
    # temperature=1,
    max_tokens=100,
    max_prompt_length=100
)
content = resp.choices[0].message.content or ""
print(content)

TypeError: Completions.create() got an unexpected keyword argument 'max_prompt_length'

# Complete Test

In [12]:
register_evaluator("math_tool_evaluator", "cookbooks.math_tool_agent.evaluator:math_tool_evaluator")

dataset_id = "math500"
dataset = DatasetRegistry.load_dataset(dataset_id, "test")
agent = load_agent("math_tool_agent")
evaluator = load_evaluator("math_tool_evaluator")

def do(process_id, idx, item, writer):
    task = Task(
        id=dataset_id,
        instruction=item["question"],
        metadata=item,
        dataset_dir=Path(f"/root/.rllm/datasets/{dataset_id}"),  # 相对路径
    )

    config = AgentConfig(
        base_url="http://localhost:30000/v1",  # LiteLLM proxy 地址
        model="/sujin/Models/Qwen/Qwen3-4B",
        session_uid="notebook-0",
    )

    episode = agent.run(task, config)
    result = evaluator.evaluate(task.metadata, episode)

    # content = episode.trajectories[0].steps[0].action
    # print(len(content))

    # from transformers import AutoTokenizer
    # tokenizer = AutoTokenizer.from_pretrained("/sujin/Models/Qwen/Qwen3-4B")
    # tokens = tokenizer.encode(content)
    # print(len(tokens))

    writer.write(f"{int(result.is_correct)}\n")


items = [item for item in dataset]
mprs = MultipleProcessRunnerSimplifier(items, do, n_process=256, return_results=True)
outputs = mprs.run()

[Errno 25] Inappropriate ioctl for device
Can't get terminal size, set terminal_y = None
7Total: [##############################################] 100% 500/500 [00:14:05 < 00:00:00, 1.69s/it]8

Aggregating results...: 256it [00:00, 17210.43it/s]


In [33]:
print(len(outputs))
results = [int(o) for o in outputs]
acc = sum(results) / len(results)
print(acc)

500
0.252


In [34]:
register_evaluator("math_tool_evaluator", "cookbooks.math_tool_agent.evaluator:math_tool_evaluator")

dataset_id = "math500"
dataset = DatasetRegistry.load_dataset(dataset_id, "test")
agent = load_agent("math_tool_agent")
evaluator = load_evaluator("math_tool_evaluator")

def do(process_id, idx, item, writer):
    task = Task(
        id=dataset_id,
        instruction=item["question"],
        metadata=item,
        dataset_dir=Path(f"/root/.rllm/datasets/{dataset_id}"),  # 相对路径
    )

    config = AgentConfig(
        base_url="http://localhost:30000/v1",  # LiteLLM proxy 地址
        model="/sujin/PycharmProjects/rllm/cookbooks/math_tool_agent/checkpoints/math_tool_agent/qwen3-4b-math_tool-verl/global_step_500/actor/huggingface",
        session_uid="notebook-0",
    )

    episode = agent.run(task, config)
    result = evaluator.evaluate(task.metadata, episode)

    # content = episode.trajectories[0].steps[0].action
    # print(len(content))

    # from transformers import AutoTokenizer
    # tokenizer = AutoTokenizer.from_pretrained("/sujin/Models/Qwen/Qwen3-4B")
    # tokens = tokenizer.encode(content)
    # print(len(tokens))

    writer.write(f"{int(result.is_correct)}\n")


items = [item for item in dataset]
mprs = MultipleProcessRunnerSimplifier(items, do, n_process=16, return_results=True)
outputs = mprs.run()

[Errno 25] Inappropriate ioctl for device
Can't get terminal size, set terminal_y = None
7Total: [##########_____________________________________] 21% 108/500 [00:02:34 < 00:09:22, 1.43s/it]8

Task The arithmetic mean of 7, 2, $x$ and 10  turn 1: LLM call failed: Error code: 400 - {'error': {'message': 'Already borrowed', 'type': 'BadRequestError', 'param': None, 'code': 400}}


7Total: [#####################__________________________] 46% 230/500 [00:06:14 < 00:07:19, 1.63s/it]8

Task What is the number of square centimeters turn 0: LLM call failed: Error code: 400 - {'error': {'message': 'Already borrowed', 'type': 'BadRequestError', 'param': None, 'code': 400}}
Process Process-527:
Traceback (most recent call last):
  File "/root/PycharmProjects/rllm/.venv/lib/python3.11/site-packages/multiprocess/process.py", line 314, in _bootstrap
    self.run()
  File "/root/PycharmProjects/rllm/.venv/lib/python3.11/site-packages/multiprocess/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/sujin/PycharmProjects/rllm/utils/mpr.py", line 407, in _target_static
    self.do(process_id, i, d, w)
  File "/tmp/ipykernel_34479/1000676735.py", line 23, in do
    result = evaluator.evaluate(task.metadata, episode)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/sujin/PycharmProjects/rllm/rllm/eval/rollout_decorator.py", line 112, in evaluate
    result = self._fn(task, episode)
             ^^^^^^^^^^^^^^^^^^^^^^^
  File "/sujin

7Total: [#####################__________________________] 46% 234/500 [00:06:21 < 00:07:13, 1.63s/it]8

...

Task Simplify $\frac{(10r^3)(4r^6)}{8r^4}$. turn 1: LLM call failed: Error code: 400 - {'error': {'message': 'Already borrowed', 'type': 'BadRequestError', 'param': None, 'code': 400}}


7Total: [##########################################_____] 91% 457/500 [00:14:19 < 00:01:20, 1.88s/it]8

Task Daniel works at an electronics store, an turn 1: LLM call failed: Error code: 400 - {'error': {'message': 'Already borrowed', 'type': 'BadRequestError', 'param': None, 'code': 400}}


7Total: [############################################___] 94% 470/500 [00:16:35 < 00:01:03, 2.12s/it]8

Aggregating results...: 16it [00:00, 8568.55it/s]


In [35]:
print(len(outputs))
results = [int(o) for o in outputs]
acc = sum(results) / len(results)
print(acc)

470
0.8765957446808511
